<a href="https://colab.research.google.com/github/yybigfish233333/chatglm_Sebastian/blob/glm4.0/sebastian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!rm -rf chatglm_Sebastian
!git clone https://github.com/yybigfish233333/chatglm_Sebastian.git
%cd chatglm_Sebastian

# 可选：绑定上游，方便以后同步（现在不需要合并）
!git remote add upstream https://github.com/ssbuild/chatglm_finetuning.git
!git fetch upstream
!git rev-parse --abbrev-ref HEAD



In [ ]:
!pip install -U -r requirements.txt
# !pip install -U git+https://github.com/ssbuild/deep_training.git
# !pip install -U "transformers>=4.30" deepspeed xformers "bitsandbytes>=0.39" "accelerate>=0.20"
!pip install -U "transformers==4.44.2" "accelerate==0.33.0" "peft==0.12.0"
!pip install -U git+https://github.com/ssbuild/deep_training.git
# 如需 deepspeed/xformers（可选）：
!pip install -U "deepspeed==0.14.4" "xformers==0.0.27"
!pip install -U "bitsandbytes==0.45.2"
!python -m bitsandbytes  # 自检，看是否显示已找到 libbitsandbytes_cuda126.so




In [ ]:
!python -m bitsandbytes  # 自检，看是否显示已找到 libbitsandbytes_cuda126.so


In [ ]:
import torch, sys
print("torch:", torch.__version__, "python:", sys.version)


In [ ]:
!pip install --force-reinstall --no-cache-dir \
  --index-url https://download.pytorch.org/whl/cu121 \
  --no-deps torchvision==0.18.1


In [ ]:
%cd /content/chatglm_Sebastian
!mkdir -p data
!cp /content/drive/MyDrive/sebastian_chat/data/finetune_train_examples.jsonl \
      data/finetune_train_examples.json   # 该仓库按“逐行 json”读取，.json/.jsonl 都可


In [ ]:
yaml_path = "/content/drive/MyDrive/sebastian_chat/config/sebastian_train_hf.yaml"
# 指向这份 YAML（关键）
import os
os.environ["train_file"] = yaml_path
print("Using config:", os.environ["train_file"])


In [ ]:
# 2) 创建输出目录（包含父级）
!mkdir -p /content/drive/MyDrive/sebastian_chat/output/sebastian_glm4_lora
!ls -ld /content/drive/MyDrive/sebastian_chat/output/sebastian_glm4_lora


In [ ]:
import os
print("train_file =", os.environ.get("train_file"))

In [ ]:
%env trainer_backend=hf
%env train_file=/content/drive/MyDrive/sebastian_chat/config/sebastian_train_hf.yaml

In [ ]:
%cd /content/chatglm_Sebastian
!python data_utils.py


In [ ]:
import os, pprint
print("ENV train_file =", os.environ.get("train_file"))

from config import config_args, global_args
print("\n=== EFFECTIVE global_args ===")
pprint.pprint(global_args)

print("\n=== EFFECTIVE lora block ===")
pprint.pprint(config_args.get("lora"))

print("\nper_device_train_batch_size =", config_args.get("per_device_train_batch_size"))
print("gradient_accumulation_steps =", config_args.get("gradient_accumulation_steps"))
print("optim =", config_args.get("optim"))


In [ ]:
!python train.py

In [ ]:
import os, glob, json, pprint, textwrap

root = "/content/drive/MyDrive/sebastian_chat/output/sebastian_glm4_lora"

print("== root files ==")
print(os.listdir(root))

# 搜索所有可能的 checkpoint 子目录
cands = [root] + sorted(glob.glob(os.path.join(root, "checkpoint-*")))
cands += [os.path.join(root, "default")]  # 有些库会把适配器放在 default/

print("\n== candidates with adapter_config.json ==")
good = []
for d in cands:
    cfg = os.path.join(d, "adapter_config.json")
    if os.path.exists(cfg):
        print("✔", d)
        good.append((d, cfg))
    else:
        print("✘", d)

# 把最可能的那个拿出来看一下
if good:
    d, cfg = good[-1]
    print("\n== peek adapter_config.json from:", d, "==")
    print(textwrap.shorten(open(cfg, "r", encoding="utf-8").read(), width=800, placeholder=" ..."))
else:
    print("\n没有任何 adapter_config.json；说明这个目录不是 peft 适配器目录，需要用正确的 checkpoint 子目录。")


In [ ]:
import os, json

roots = [
    "/content/drive/MyDrive/sebastian_chat/output/sebastian_glm4_lora",
    "/content/drive/MyDrive/sebastian_chat/output/sebastian_glm4_lora/checkpoint-22",
]

for d in roots:
    cfg_path = os.path.join(d, "adapter_config.json")
    if not os.path.exists(cfg_path):
        print("skip (no file):", cfg_path)
        continue
    with open(cfg_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    changed = False
    if "peft_type" not in cfg:
        cfg["peft_type"] = "LORA"; changed = True
    if "task_type" not in cfg:
        cfg["task_type"] = "CAUSAL_LM"; changed = True

    # 可保持和训练时一致（你有效配置里 r=8, dropout=0.1, alpha=32, target_modules=["query_key_value"]）
    cfg.setdefault("r", 8)
    cfg.setdefault("lora_alpha", 32)
    cfg.setdefault("lora_dropout", 0.1)
    cfg.setdefault("bias", "none")
    cfg.setdefault("target_modules", ["query_key_value"])

    if changed:
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, ensure_ascii=False, indent=2)
        print("patched:", cfg_path)
    else:
        print("already ok:", cfg_path)


In [ ]:
import os, json

roots = [
    "/content/drive/MyDrive/sebastian_chat/output/sebastian_glm4_lora",
    "/content/drive/MyDrive/sebastian_chat/output/sebastian_glm4_lora/checkpoint-22",
]

# Peft 的 LoraConfig 支持的关键字段（0.12.x）
allowed = {
    "r","lora_alpha","lora_dropout","target_modules","bias","task_type",
    "inference_mode","fan_in_fan_out","modules_to_save","layers_to_transform",
    "layers_pattern","rank_pattern","alpha_pattern","init_lora_weights",
    # 下面这个不是必须，但留着也无害
    "peft_type"
}

for d in roots:
    cfg_path = os.path.join(d, "adapter_config.json")
    if not os.path.exists(cfg_path):
        print("skip (no file):", cfg_path)
        continue

    with open(cfg_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    # 补齐 peft 需要的字段
    raw.setdefault("peft_type", "LORA")
    raw.setdefault("task_type", "CAUSAL_LM")
    raw.setdefault("bias", "none")
    raw.setdefault("r", 8)
    raw.setdefault("lora_alpha", 32)
    raw.setdefault("lora_dropout", 0.1)
    raw.setdefault("inference_mode", True)
    raw.setdefault("target_modules", ["query_key_value"])

    # 仅保留 peft 支持的键，去掉 enable / lora_type / with_lora 等自定义键
    cleaned = {k: v for k, v in raw.items() if k in allowed}

    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2)

    print("patched:", cfg_path, "-> keys:", sorted(cleaned.keys()))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

base = "THUDM/glm-4-9b-chat"
# 用具体 checkpoint，确保拿到最新权重
lora_dir = "/content/drive/MyDrive/sebastian_chat/output/sebastian_glm4_lora/checkpoint-22"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(base, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    base, trust_remote_code=True, device_map="auto", quantization_config=bnb
).eval()

# 载入 LoRA（你之前已经把 adapter_config.json 清洗过了）
model = PeftModel.from_pretrained(base_model, lora_dir).eval()

# —— 关键：用训练时的 system 设定 + chat 模板 —— #
messages = [
    {"role": "system", "content": "你是《星露谷物语》的 NPC——塞巴斯蒂安。回答要贴合他的人设：内向、略冷、宅气、程序员气质；尽量简短直接，不要编造设定外信息。"},
    {"role": "user", "content": "你知道鹈鹕镇吗？我刚刚搬来这里。"},
]

# 由分词器生成带特殊token的对话串，并自动加上“待生成的 assistant”前缀
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,                 # 先用贪心，便于验证
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.05,
        no_repeat_ngram_size=3,
    )

# 只取新生成的部分（去掉提示词）
gen = out[0, inputs["input_ids"].shape[-1]:]
print(tokenizer.decode(gen, skip_special_tokens=True))
